In [9]:
import sys
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import NumericType
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
sys.path.append(os.path.abspath(".."))
from algorithms.fpgrowth import FPGrowthAssociation
from algorithms.clustering import KMeansSegmentation


spark = SparkSession.builder \
    .appName("Weather_Preprocessing_Jupyter") \
    .master("local[*]") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()

In [10]:
 #? CHECK NULL/NaN 
def check_missing(df):
    missing_exprs = []
    # Tạo danh sách các biểu thức kiểm tra
    for c, dtype in df.dtypes:
        is_null_col = F.col(c).isNull()
        # Nếu là kiểu số (Float/Double), kiểm tra thêm NaN
        if dtype in ("double", "float"):
            is_missing = is_null_col | F.isnan(c)
        else:
            is_missing = is_null_col
        # Đếm số lượng thỏa mãn điều kiện
        missing_exprs.append(F.count(F.when(is_missing, c)).alias(c))
    print("--- Đang kiểm tra Null/NaN trên từng cột... ---")
    
    # Thực thi lệnh và lấy kết quả dưới dạng Dictionary
    missing_counts = df.select(missing_exprs).collect()[0].asDict()
    # Hiển thị kết quả ra màn hình
    print(f"{'Cột':<25} | {'Số lượng Null/NaN':<20}")
    print("-" * 50)
    for col_name, count in missing_counts.items():
        print(f"{col_name:<25} | {count:<20}")
    return missing_counts
 #? MIN, MAX, Count <=0 của Price và Quantity
def get_min_max_numerical(df):
    """
    Tìm giá trị Min, Max và đếm số lượng giá trị <= 0 cho tất cả các cột định lượng.
    """
    # 1. Tự động lấy danh sách cột số
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
    
    print(f"--- Phân tích chuyên sâu các cột số: {numeric_cols} ---")
    
    # 2. Tạo danh sách các phép tính aggregation cho từng cột
    # Chúng ta sẽ tính Min, Max và Count (giá trị <= 0)
    agg_exprs = []
    for col_name in numeric_cols:
        agg_exprs.append(F.min(F.col(col_name)).alias(f"{col_name}_min"))
        agg_exprs.append(F.max(F.col(col_name)).alias(f"{col_name}_max"))
        agg_exprs.append(F.count(F.when(F.col(col_name) <= 0, 1)).alias(f"{col_name}_le_zero"))

    # 3. Chạy aggregation một lần duy nhất để tối ưu hiệu năng
    summary_dict = df.agg(*agg_exprs).collect()[0].asDict()

    # 4. Chuyển đổi kết quả sang dạng bảng (Pandas) để in ra cho đẹp
    rows = []
    for col_name in numeric_cols:
        rows.append({
            "Column": col_name,
            "Min": summary_dict[f"{col_name}_min"],
            "Max": summary_dict[f"{col_name}_max"],
            "Count_<=_0": summary_dict[f"{col_name}_le_zero"]
        })
    
    import pandas as pd
    summary_df = pd.DataFrame(rows)
    
    # Hiển thị kết quả
    print(summary_df.to_string(index=False))
    
    return summary_df
 #? Loại bỏ dòng trùng lặp
def remove_duplicates(df: DataFrame, subset_cols: list = None) -> DataFrame:
    """
    Loại bỏ các dòng dữ liệu trùng lặp trong PySpark DataFrame.
    
    :param df: PySpark DataFrame đầu vào.
    :param subset_cols: (Tùy chọn) Danh sách các cột dùng để xét trùng lặp. 
                        Nếu để None, Spark sẽ xét trùng lặp trên TOÀN BỘ các cột.
    :return: DataFrame đã sạch bóng dữ liệu trùng.
    """
    print("--- Đang kiểm tra và xử lý dữ liệu trùng lặp... ---")
    
    # Đếm số lượng dòng trước khi xử lý
    count_before = df.count()
    
    # Xử lý trùng lặp
    if subset_cols:
        print(f" > Tiêu chí xét trùng lặp: Dựa trên các cột {subset_cols}")
        df_clean = df.dropDuplicates(subset=subset_cols)
    else:
        print(" > Tiêu chí xét trùng lặp: Xét trên toàn bộ các cột (Exact Match)")
        df_clean = df.dropDuplicates()
        
    # Đếm số lượng dòng sau khi xử lý
    count_after = df_clean.count()
    duplicates_removed = count_before - count_after
    
    # Báo cáo kết quả
    print(f" > Số dòng ban đầu: {count_before}")
    print(f" > Số dòng sau khi dọn dẹp: {count_after}")
    
    if duplicates_removed > 0:
        print(f" [+] Thành công! Đã phát hiện và xóa {duplicates_removed} dòng trùng lặp.")
    else:
        print(" [v] Dữ liệu rất sạch, không phát hiện dòng trùng lặp nào!")
        
    print("-" * 50)
    return df_clean

#? Xử lý ngoại lai và số âm/không cho TỪNG SẢN PHẨM
def clean_outliers_smart_impute(df, quantity_col="Quantity", price_col="Price", group_col="ProductNo"):
    """
    Xử lý toàn diện:
    1. Xử lý Quantity: Bỏ <= 0, khử ngoại lai bằng Median.
    2. Xử lý Price: Thay thế <= 0 và ngoại lai bằng Mean (chỉ tính Mean từ giá dương).
    """
    
    # --- BƯỚC 0: Xử lý trùng lặp để tránh làm lệch thống kê ---
    df = df.dropDuplicates()

    # --- BƯỚC 1: Lọc bỏ Quantity lỗi (Đơn hủy/trả thường không dùng để tính Mean/Median bán hàng) ---
    # Lưu ý: Ta chưa lọc Price <= 0 vì sẽ bù đắp (impute) nó sau.
    df_filtered = df.filter(F.col(quantity_col) > 0)
    
    print(f"--- BƯỚC 2: Tính toán chỉ số thống kê (Window) theo từng {group_col} ---")
    window_spec = Window.partitionBy(group_col)
    
    # Chỉ tính toán dựa trên các giá trị hợp lệ (Price > 0)
    valid_price_cond = F.when(F.col(price_col) > 0, F.col(price_col))

    df_stats = df_filtered \
        .withColumn("q1_q", F.percentile_approx(F.col(quantity_col), 0.25).over(window_spec)) \
        .withColumn("q3_q", F.percentile_approx(F.col(quantity_col), 0.75).over(window_spec)) \
        .withColumn("median_q", F.percentile_approx(F.col(quantity_col), 0.5).over(window_spec)) \
        .withColumn("q1_p", F.percentile_approx(valid_price_cond, 0.25).over(window_spec)) \
        .withColumn("q3_p", F.percentile_approx(valid_price_cond, 0.75).over(window_spec)) \
        .withColumn("mean_p", F.avg(valid_price_cond).over(window_spec))

    # --- BƯỚC 3: Tính toán biên IQR cho cả Quantity và Price ---
    df_bounds = df_stats \
        .withColumn("iqr_q", F.col("q3_q") - F.col("q1_q")) \
        .withColumn("lower_q", F.greatest(F.lit(1), F.col("q1_q") - 1.5 * F.col("iqr_q"))) \
        .withColumn("upper_q", F.col("q3_q") + 1.5 * F.col("iqr_q")) \
        .withColumn("iqr_p", F.col("q3_p") - F.col("q1_p")) \
        .withColumn("lower_p", F.col("q1_p") - 1.5 * F.col("iqr_p")) \
        .withColumn("upper_p", F.col("q3_p") + 1.5 * F.col("iqr_p"))

    # --- BƯỚC 4: Thực hiện thay thế (Imputation) ---
    df_final = df_bounds \
        .withColumn(
            quantity_col,
            F.when(
                (F.col(quantity_col) < F.col("lower_q")) | (F.col(quantity_col) > F.col("upper_q")),
                F.coalesce(F.col("median_q"), F.lit(1))
            ).otherwise(F.col(quantity_col))
        ) \
        .withColumn(
            price_col,
            F.when(
                (F.col(price_col) <= 0) | # Giá trị âm hoặc bằng 0
                (F.col(price_col) < F.col("lower_p")) | # Ngoại lai dưới
                (F.col(price_col) > F.col("upper_p")),  # Ngoại lai trên
                F.coalesce(F.col("mean_p"), F.lit(0)) # Thay bằng Mean, nếu không có Mean thì để 0
            ).otherwise(F.col(price_col))
        )

    # Loại bỏ các cột tạm
    temp_cols = [c for c in df_final.columns if "_q" in c or "_p" in c or "mean" in c or "median" in c or "iqr" in c]
    
    print(" [+] Hoàn tất: Xử lý Quantity (Median) và Price (Mean) thành công!")
    return df_final.drop(*temp_cols)


 # ? Tạo dữ liệu cấp độ Giao dịch (Basket) và phân khúc đơn hàng
def create_transaction_level_data(df):
    """
    1. Gom nhóm theo TransactionNo để tạo Basket.
    2. Tính tổng giá trị đơn hàng (TotalValue).
    3. Phân khúc giỏ hàng dựa trên TotalValue.
    """
    print("--- Đang chuyển đổi sang dữ liệu cấp độ Giao dịch (Basket)... ---")
    # Bước 1: Tính thành tiền cho từng dòng trước khi gom
    df_with_line_total = df.withColumn("LineTotal", F.col("Price") * F.col("Quantity"))
    # Bước 2: GroupBy để tạo Basket và tính tổng tiền đơn hàng
    # Gom các cột thông tin chung của giao dịch vào groupBy
    df_basket = df_with_line_total.groupBy("TransactionNo", "Date", "CustomerNo", "Country") \
        .agg(
            F.round(F.sum("LineTotal"), 2).alias("TotalOrderValue"),
            F.collect_set("ProductName").alias("Basket"),
            F.count("ProductNo").alias("TotalItems")
        )
    # Bước 3: Tạo Price Segment dựa trên tổng giá trị đơn hàng (TotalOrderValue)
    # Lưu ý: Sử dụng F.lit() để tránh lỗi UNRESOLVED_COLUMN đã gặp
    df_final = df_basket.withColumn(
        "Order_segment",
        F.when(F.col("TotalOrderValue") < 50, F.lit("Small-Order"))
         .when((F.col("TotalOrderValue") >= 50) & (F.col("TotalOrderValue") <= 300), F.lit("Medium-Order"))
         .otherwise(F.lit("Large-Order"))
    )
    return df_final

 # ? One-Hot Encoding cho cột định tính
def apply_one_hot_encoding(df, categorical_cols=["Country", "Order_segment"]):
    """
    Bản nâng cấp: 
    1. Ngắt phả hệ (Lineage) để tránh sập Driver.
    2. Kiểm tra số lượng giá trị duy nhất (tránh lỗi Encoder).
    3. Xử lý giá trị NULL (tránh lỗi Indexer).
    """
    print(f"--- Đang chuẩn bị One-Hot Encoding cho: {categorical_cols} ---")

    # BƯỚC 1: CỰC KỲ QUAN TRỌNG - Ngắt phả hệ để Spark không bị quá tải
    # Dùng localCheckpoint để "chốt" dữ liệu sạch trước khi vào Pipeline
    df = df.localCheckpoint()

    # BƯỚC 2: Kiểm tra dữ liệu thực tế trước khi tạo Stage
    valid_stages = []
    for col in categorical_cols:
        # Kiểm tra số lượng giá trị duy nhất (phải >= 2 mới Encode được)
        distinct_count = df.select(col).distinct().count()
        
        # Kiểm tra xem có Null không
        null_count = df.filter(F.col(col).isNull()).count()
        
        if distinct_count < 2:
            print(f" [!] Bỏ qua cột '{col}' vì chỉ có {distinct_count} giá trị duy nhất (OneHot không chạy được).")
            continue

        if null_count > 0:
            print(f" [!] Cảnh báo: Cột '{col}' có {null_count} dòng Null. Sẽ được nhóm vào một category riêng.")

        # 1. Chuyển chữ -> số (Thêm handleInvalid để không bị sập nếu gặp Null hoặc giá trị mới)
        indexer = StringIndexer(inputCol=col, outputCol=f"{col}_index", handleInvalid="keep")
        
        # 2. Chuyển số -> Vector
        encoder = OneHotEncoder(inputCol=f"{col}_index", outputCol=f"{col}_ohe")
        
        valid_stages += [indexer, encoder]

    if not valid_stages:
        print(" [X] Không có cột nào đủ điều kiện để Encode. Trả về DF gốc.")
        return df

    # BƯỚC 3: Chạy Pipeline
    print(f" [+] Đang thực thi Pipeline với {len(valid_stages)} stages...")
    pipeline = Pipeline(stages=valid_stages)
    
    try:
        model = pipeline.fit(df)
        df_encoded = model.transform(df)
        
        # Xóa các cột index trung gian để nhẹ data
        cols_to_drop = [f"{col}_index" for col in categorical_cols if f"{col}_index" in df_encoded.columns]
        df_final = df_encoded.drop(*cols_to_drop)
        
        print(" [+] Mã hóa hoàn tất.")
        return df_final
        
    except Exception as e:
        print(f" [!] Lỗi khi chạy Pipeline: {str(e)}")
        return df

In [11]:
 #! ĐỌC FILE VÀ CHẠY TOÀN BỘ QUÁ TRÌNH
input_file = '../data.parquet'
print(f"Đang đọc dữ liệu từ '{input_file}'...")
df = spark.read.parquet(input_file)
print(df.count())
# ! EDA
# weekly_trend = eda_temporal_trends(df, time_col="Date")
# topproduct = eda_top_products(df, item_col="ProductName")
#! Tiền xử lý dữ liệu
report = check_missing(df)
stats= get_min_max_numerical(df)


df = remove_duplicates(df)
df = clean_outliers_smart_impute(df, "Quantity", "Price", "ProductNo")



print(f"Tổng số dòng trong sau khi xử lý: {df.count()}")
df = create_transaction_level_data(df)
df = apply_one_hot_encoding(df)
print(f"Tổng số dòng giao dịch sau khi merge lại {df.count()}")
print(df.show(5))

Đang đọc dữ liệu từ '../data.parquet'...
1609050
--- Đang kiểm tra Null/NaN trên từng cột... ---


Cột                       | Số lượng Null/NaN   
--------------------------------------------------
TransactionNo             | 0                   
Date                      | 0                   
ProductNo                 | 0                   
ProductName               | 0                   
Price                     | 0                   
Quantity                  | 0                   
CustomerNo                | 0                   
Country                   | 0                   
--- Phân tích chuyên sâu các cột số: ['Price', 'Quantity'] ---
  Column       Min      Max  Count_<=_0
   Price      5.13   660.62           0
Quantity -80995.00 80995.00       25755
--- Đang kiểm tra và xử lý dữ liệu trùng lặp... ---
 > Tiêu chí xét trùng lặp: Xét trên toàn bộ các cột (Exact Match)


 > Số dòng ban đầu: 1609050
 > Số dòng sau khi dọn dẹp: 1593450
 [+] Thành công! Đã phát hiện và xóa 15600 dòng trùng lặp.
--------------------------------------------------
--- BƯỚC 2: Tính toán chỉ số thống kê (Window) theo từng ProductNo ---
 [+] Hoàn tất: Xử lý Quantity (Median) và Price (Mean) thành công!


Tổng số dòng trong sau khi xử lý: 1567806
--- Đang chuyển đổi sang dữ liệu cấp độ Giao dịch (Basket)... ---
--- Đang chuẩn bị One-Hot Encoding cho: ['Country', 'Order_segment'] ---


 [+] Đang thực thi Pipeline với 4 stages...
 [+] Mã hóa hoàn tất.
Tổng số dòng giao dịch sau khi merge lại 59370
+-------------+---------+----------+--------------+---------------+--------------------+----------+-------------+--------------+-----------------+
|TransactionNo|     Date|CustomerNo|       Country|TotalOrderValue|              Basket|TotalItems|Order_segment|   Country_ohe|Order_segment_ohe|
+-------------+---------+----------+--------------+---------------+--------------------+----------+-------------+--------------+-----------------+
|       536365|12/1/2018|     17850|United Kingdom|         560.85|[White Moroccan M...|         7|  Large-Order|(38,[0],[1.0])|    (3,[0],[1.0])|
|       536392|12/1/2018|     13705|United Kingdom|        1381.83|[3 Stripey Mice F...|        10|  Large-Order|(38,[0],[1.0])|    (3,[0],[1.0])|
|       536393|12/1/2018|     13747|United Kingdom|           40.9|    [Retrospot Lamp]|         1|  Small-Order|(38,[0],[1.0])|    (3,[2],[1.0])|
|    

In [12]:
def save_to_parquet(df: DataFrame, output_path: str, mode: str = "overwrite", single_file: bool = True):
    """
    Hàm lưu PySpark DataFrame thành định dạng Parquet một cách an toàn.
    
    Tham số:
    - df (DataFrame): Dữ liệu cần lưu.
    - output_path (str): Đường dẫn lưu file (VD: "../data/my_data.parquet").
    - mode (str): Chế độ lưu ("overwrite" - ghi đè, "append" - ghi nối tiếp, "error" - báo lỗi nếu file đã tồn tại).
    - single_file (bool): Nếu True, gom toàn bộ dữ liệu về 1 file Part duy nhất bằng coalesce(1).
    """
    try:
        print(f"--- Đang chuẩn bị lưu dữ liệu ra file Parquet... ---")
        print(f" > Đường dẫn đích: {output_path}")
        print(f" > Chế độ lưu (Mode): '{mode}'")
        
        # Gom dữ liệu về 1 phân vùng nếu được yêu cầu
        df_to_save = df.coalesce(1) if single_file else df
        
        # Thực thi lệnh ghi
        df_to_save.write.mode(mode).parquet(output_path)
        
        print(" [+] Hoàn tất! File Parquet đã được lưu thành công.")
        print("-" * 50)
        
    except Exception as e:
        print(f" [!] Lỗi trong quá trình lưu file: {str(e)}")


save_to_parquet(df, "../data/results/final_dataset_streaming.parquet")


--- Đang chuẩn bị lưu dữ liệu ra file Parquet... ---
 > Đường dẫn đích: ../data/results/final_dataset_streaming.parquet
 > Chế độ lưu (Mode): 'overwrite'


 [+] Hoàn tất! File Parquet đã được lưu thành công.
--------------------------------------------------


In [13]:
 # ! Bài toán Association Rules
def export_association_data(df_transactions):
    """
    Chuẩn bị dữ liệu cho bài toán Association Rules từ DataFrame đã gom nhóm.
    Dữ liệu đầu vào (df_transactions) có schema:
    |TransactionNo|TotalOrderValue|Basket|TotalItems|...|
    """
    print(f"--- Đang chuẩn bị dữ liệu xuất cho Association Rules... ---")

    # 1. Xử lý giỏ hàng theo Sản phẩm (Dùng cột 'Basket' có sẵn)
    # Ta chỉ lấy những đơn có từ 2 món trở lên (size > 1) để tìm luật kết hợp
    basket_product = df_transactions.filter(F.size(F.col("Basket")) >= 1) \
                                    .select(
                                        F.col("TransactionNo"), # Giữ lại cột TransactionNo
                                        F.col("CustomerNo"),    # Giữ lại cột CustomerNo
                                        F.col("Basket").alias("items")
                                    )
    # 3. Xuất file Parquet
    print(" > Đang xuất file asso_products.parquet...")
    basket_product.write.mode("overwrite").parquet("../asso_products.parquet")

    print("-" * 50)
    print("Thống kê dữ liệu đã lọc (Size >= 1):")
    print(f" - Số lượng giỏ hàng Sản phẩm: {basket_product.count()}")    
    return basket_product

basket_p = export_association_data(df)
print(len(basket_p.columns))
# 1. Khởi tạo mô hình (Nó sẽ tự lo việc cấu hình checkpoint và đọc file)
fp_model = FPGrowthAssociation(spark, input_path="../asso_products.parquet")

# 2. Huấn luyện và tự động xuất ra file parquet
rules_df = fp_model.train_and_save(
    output_rules_path="../data/results/association_rules.parquet", 
    min_support=0.025, 
    min_confidence=0.5,
    min_lift=2.0
)
# 3. Hiển thị kết quả bằng các hàm tích hợp sẵn
fp_model.display_top_itemsets(top_n=100)
fp_model.display_top_rules(top_n=100)

print(rules_df.count())


--- Đang chuẩn bị dữ liệu xuất cho Association Rules... ---
 > Đang xuất file asso_products.parquet...


--------------------------------------------------
Thống kê dữ liệu đã lọc (Size >= 1):
 - Số lượng giỏ hàng Sản phẩm: 59370
3
--- Đang đọc dữ liệu từ ../asso_products.parquet... ---
--- Đang huấn luyện FP-Growth... ---


--- Đang tiến hành lưu luật (37 luật) ---
 [+] Đã lưu Parquet tại: ../data/results/association_rules.parquet
 [+] Đã lưu CSV tại: ../data/results/association_rules.csv
 [+] Đã lưu JSON tại: ../data/results/association_rules.json
 TOP 100 TẬP MỤC PHỔ BIẾN =============================
                            itemsets                              freq  support
                              Cream Hanging Heart T-Light Holder  6807  0.1147 
                                         Jumbo Bag Red Retrospot  6276  0.1057 
                                        Regency Cakestand 3 Tier  5967  0.1005 
                                                   Party Bunting  5061  0.0852 
                                         Lunch Bag Red Retrospot  4701  0.0792 
                                   Assorted Colour Bird Ornament  4371  0.0736 
                                                  Popcorn Holder  4176  0.0703 
                                Set Of 3 Cake Tins Pantry Design  4155  0.0

In [14]:
def build_retail_user_profile(df, output_parquet_path):
    """
    Nhận DataFrame giao dịch đã làm sạch, xử lý định dạng ngày tháng,
    gom nhóm thành User Profile (RFM & Hành vi), và ghi ra file Parquet.
    """
    print("--- 1. Đang tiếp nhận DataFrame để xây dựng User Profile... ---")
    # BƯỚC SỬA LỖI: Ép kiểu cột Date từ String (M/d/yyyy) sang DateType chuẩn của Spark
    # Dùng to_date thay vì to_timestamp vì chúng ta chỉ cần ngày để tính datediff
    df = df.withColumn("Date", F.to_date(F.col("Date"), "M/d/yyyy"))

    # Bỏ qua các giao dịch không có mã khách hàng hoặc Date bị null (do sai format)
    valid_df = df.filter(
        F.col("CustomerNo").isNotNull() & 
        (F.col("TotalOrderValue") > 0) &
        F.col("Date").isNotNull()
    )
    # Lấy mốc thời gian mới nhất trong hệ thống để tính Recency
    # Do đã ép kiểu ở trên, max_date_row bây giờ sẽ là một object datetime.date chuẩn
    max_date_row = valid_df.select(F.max("Date")).collect()[0][0]

    print("--- 2. Đang tính toán RFM và Thói quen mua sắm... ---")
    
    # BƯỚC A: Tính R, F, M cơ bản
    rfm = valid_df.groupBy("CustomerNo").agg(
        F.round(F.sum("TotalOrderValue"), 2).alias("total_spend"),     
        F.countDistinct("TransactionNo").alias("total_orders"),        
        F.max("Date").alias("last_purchase_date"),                     
        F.round(F.avg("TotalItems"), 2).alias("avg_items_per_order")   
    )

    # BƯỚC B: Tìm phân khúc giỏ hàng khách hay mua nhất (Mode Order_segment)
    cat_counts = valid_df.groupBy("CustomerNo", "Order_segment").count()
    windowSpec = Window.partitionBy("CustomerNo").orderBy(F.col("count").desc())
    
    favorite_segment = cat_counts.withColumn("rn", F.row_number().over(windowSpec))\
                                 .filter(F.col("rn") == 1)\
                                 .select("CustomerNo", F.col("Order_segment").alias("favorite_order_size"))

    # BƯỚC C: Tìm Quốc gia chính của khách hàng
    country_counts = valid_df.groupBy("CustomerNo", "Country").count()
    main_country = country_counts.withColumn("rn", F.row_number().over(windowSpec))\
                                 .filter(F.col("rn") == 1)\
                                 .select("CustomerNo", F.col("Country").alias("main_country"))

    # BƯỚC D: Gộp các đặc trưng lại với nhau
    df_users = rfm.join(favorite_segment, "CustomerNo", "left") \
                  .join(main_country, "CustomerNo", "left")

    # BƯỚC E: Hoàn thiện các chỉ số cuối cùng (Tính Recency)
    # datediff bây giờ sẽ hoạt động hoàn hảo vì cả 2 vế đều là chuẩn Date
    df_final = df_users.withColumn(
        "recency_days", F.datediff(F.lit(max_date_row), F.col("last_purchase_date"))
    ).withColumn(
        "recency_days", F.when(F.col("recency_days") < 0, 0).otherwise(F.col("recency_days"))
    ).withColumn(
        "avg_order_value", F.round(F.col("total_spend") / F.col("total_orders"), 2)
    ).drop("last_purchase_date")

    # Ghi ra file Parquet
    print(f"--- 3. Đang xuất file Parquet User Profile ra: {output_parquet_path} ---")
    df_final.write.mode("overwrite").parquet(output_parquet_path)
    
    print(" [+] Hoàn tất! File đã sẵn sàng để đưa vào mô hình Clustering.")
    
    return df_final

# Giả sử 'df' là DataFrame bạn vừa chạy xong các bước làm sạch (xóa giá trị âm, lọc null...)
output_path = "../data/processed/user_profile.parquet"
# Gọi hàm
df_user_profile = build_retail_user_profile(df, output_path)

# 1. Khởi tạo (Nó sẽ tự động dọn dẹp lỗi null, âm, và ngắt phả hệ)
seg_model = KMeansSegmentation(spark, "../data/processed/user_profile.parquet")

# 2. Đánh giá (Tùy chọn)
print("\n--- Tiêu chí chọn K: Silhouette càng cao càng tốt, WCSS có điểm gãy (Elbow) ---")
metrics = seg_model.evaluate_k(range(2, 9))

# 3. Huấn luyện và Lưu
df_clustered = seg_model.train_and_save("../data/results/clustered_users.parquet", n_clusters=4)

# Xem thử kết quả
df_clustered.show()

--- 1. Đang tiếp nhận DataFrame để xây dựng User Profile... ---


--- 2. Đang tính toán RFM và Thói quen mua sắm... ---
--- 3. Đang xuất file Parquet User Profile ra: ../data/processed/user_profile.parquet ---


 [+] Hoàn tất! File đã sẵn sàng để đưa vào mô hình Clustering.
--- Đọc dữ liệu từ ../data/processed/user_profile.parquet... ---
--- [Auto-Clean] Đang dọn dẹp dữ liệu để tránh lỗi K-Means... ---
--- [Auto-Clean] Hoàn tất dọn dẹp! ---

--- Tiêu chí chọn K: Silhouette càng cao càng tốt, WCSS có điểm gãy (Elbow) ---
--- Đang tiền xử lý dữ liệu (Log + Scaling)... ---
--- Bắt đầu đánh giá các giá trị K... ---


26/04/28 01:14:59 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


 > K= 2 | WCSS:        35359.31 | Silhouette Score: 0.5131


 > K= 3 | WCSS:        26705.39 | Silhouette Score: 0.4474


 > K= 4 | WCSS:        20279.25 | Silhouette Score: 0.4915
 > K= 5 | WCSS:        16569.06 | Silhouette Score: 0.4727
 > K= 6 | WCSS:        14700.86 | Silhouette Score: 0.4499


 > K= 7 | WCSS:        13582.64 | Silhouette Score: 0.4068
 > K= 8 | WCSS:        11867.86 | Silhouette Score: 0.4395
--- Đang huấn luyện KMeans với K=4... ---
+-----------+-----------+------------+-------------------+-------------------+--------------+------------+---------------+-------+
|CustomerNo |total_spend|total_orders|avg_items_per_order|favorite_order_size|main_country  |recency_days|avg_order_value|cluster|
+-----------+-----------+------------+-------------------+-------------------+--------------+------------+---------------+-------+
|14211      |11397.54   |9           |12.44              |Large-Order        |United Kingdom|56          |1266.39        |3      |
|16013      |19154.84   |48          |6.4                |Medium-Order       |United Kingdom|6           |399.06         |3      |
|COPY1_12877|12251.07   |12          |12.83              |Large-Order        |United Kingdom|9           |1020.92        |3      |
|COPY1_18109|20727.09   |26          |16.92           